# Story Writing Benchmark — Analysis Notebook

Evaluates alignment between open-source judges and the Claude Sonnet 4.6 proprietary baseline on the `lars1234/story_writing_benchmark` dataset (3,480 stories, 15 metrics, 0–5 scale, all positive).

**Sections**
1. Load data
2. Alignment metrics — summary table (MAE, RMSE, Spearman ρ, Kendall τ)
3. Per-category MAE heatmap
4. Cost analysis — throughput & cost/1M tokens
5. MOJO cost analysis — blended cost at key routing thresholds

## Setup

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats
from IPython.display import display

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["figure.dpi"] = 120

RESULTS_DIR   = Path("../dataset/results")
LOGS_DIR      = Path("../logs")
BASELINE_FILE = "claude_sonnet_4.6.csv"

GPU_RATE_LOW  = 2.0   # $/hr (H100)
GPU_RATE_HIGH = 3.5
CLAUDE_PRICE_INPUT  = 3.0   # $/1M input tokens
CLAUDE_PRICE_OUTPUT = 15.0  # $/1M output tokens

ALL_CATEGORIES = [
    "Grammar, Spelling, and Punctuation Quality",
    "Clarity and Understandability",
    "Logical Connection Between Events and Ideas",
    "Scene Construction and Purpose",
    "Internal Consistency",
    "Character Consistency",
    "Character Motivation and Actions",
    "Sentence Pattern Variety",
    "Avoidance of Clichés and Overused Phrases",
    "Natural Dialogue",
    "Avoidance of Predictable Narrative Tropes",
    "Character Depth and Dimensionality",
    "Realistic Character Interactions",
    "Ability to Hold Reader Interest",
    "Satisfying Plot Resolution",
]
ALL_COLS = [f"{c}_score" for c in ALL_CATEGORIES]
KEYS     = ["index", "prompt_id", "model"]

def pretty_model(stem: str) -> str:
    s = stem.replace("_swb_result", "").replace("_result", "")
    s = s.replace("-Instruct-2507", "").replace("-Instruct", "")
    s = s.replace("_it", "").replace("-it", "")
    s = s.replace("-BF16", "")
    return s

## 1. Load data

In [ ]:
ref = pd.read_csv(RESULTS_DIR / BASELINE_FILE)

evaluator_dfs = {}
for p in sorted(RESULTS_DIR.glob("*.csv")):
    if "claude" in p.name:
        continue
    name = pretty_model(p.stem)
    ev = pd.read_csv(p)
    m = pd.merge(ev, ref, on=KEYS, suffixes=("_ev", "_ref"), how="inner")
    evaluator_dfs[name] = m

print(f"Baseline: {BASELINE_FILE}  ({len(ref)} rows)")
for name, df in evaluator_dfs.items():
    print(f"  {name}: {len(df)} rows")

## 2. Alignment metrics

Per-model MAE, RMSE (absolute error vs baseline) and Spearman ρ, Kendall τ (rank correlation).

In [ ]:
summary_rows = []

for name, m in evaluator_dfs.items():
    all_diffs, all_ev, all_ref_vals = [], [], []

    for col in ALL_COLS:
        ev_s  = m[f"{col}_ev"].to_numpy(dtype=float)
        ref_s = m[f"{col}_ref"].to_numpy(dtype=float)
        mask  = ~(np.isnan(ev_s) | np.isnan(ref_s))
        diff  = ev_s[mask] - ref_s[mask]
        all_diffs.append(diff)
        all_ev.extend(ev_s[mask])
        all_ref_vals.extend(ref_s[mask])

    all_d = np.concatenate(all_diffs)
    rho, _ = stats.spearmanr(all_ev, all_ref_vals)
    tau, _ = stats.kendalltau(all_ev, all_ref_vals)

    summary_rows.append({
        "Model":       name,
        "MAE":         round(float(np.mean(np.abs(all_d))), 3),
        "RMSE":        round(float(np.sqrt(np.mean(all_d**2))), 3),
        "Spearman ρ":  round(float(rho), 3),
        "Kendall τ":   round(float(tau), 3),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("MAE").reset_index(drop=True)
display(summary_df)

print("\nLaTeX table rows:")
for _, r in summary_df.iterrows():
    print(f"{r['Model']:<35} & {r['MAE']:.3f} & {r['RMSE']:.3f} & {r['Spearman ρ']:.3f} & {r['Kendall τ']:.3f} \\\\")

## 3. Per-category MAE heatmap

In [ ]:
per_cat_rows = []
model_order = summary_df["Model"].tolist()

for name, m in evaluator_dfs.items():
    for col in ALL_COLS:
        ev_s  = m[f"{col}_ev"].to_numpy(dtype=float)
        ref_s = m[f"{col}_ref"].to_numpy(dtype=float)
        mask  = ~(np.isnan(ev_s) | np.isnan(ref_s))
        diff  = ev_s[mask] - ref_s[mask]
        per_cat_rows.append({
            "model":  name,
            "rubric": col.replace("_score", ""),
            "mae":    float(np.mean(np.abs(diff))),
        })

per_cat_df = pd.DataFrame(per_cat_rows)
pivot = per_cat_df.pivot(index="rubric", columns="model", values="mae")[model_order]
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
            linewidths=0.4, cbar_kws={"label": "MAE", "shrink": 0.7})
ax.set_title("Per-category MAE vs Claude Sonnet 4.6 (SWB)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 4. Cost analysis

Computes throughput (tok/s) and effective cost per 1M tokens from inference logs.

$$\text{Throughput} = \frac{\sum \text{total\_tokens}}{\sum \text{latency\_s}} \qquad \text{Cost}_{1\text{M}} = \frac{10^6}{\text{Throughput} \times 3600} \times \text{Rate}_{\text{GPU}}$$

In [ ]:
def parse_log(path: Path) -> dict:
    tot = lat = prompt = completion = n_ok = 0
    with open(path) as f:
        for line in f:
            e = json.loads(line)
            if e.get("status") == "ok":
                tot        += e["total_tokens"]
                lat        += e["latency_ms"] / 1000
                prompt     += e["prompt_tokens"]
                completion += e["completion_tokens"]
                n_ok += 1
    return dict(total_tokens=tot, latency_s=lat,
                prompt_tokens=prompt, completion_tokens=completion, n_ok=n_ok)

# Claude baseline
claude_raw = parse_log(LOGS_DIR / "llm_calls_claude-sonnet-4-6.jsonl")
claude_cost_total = (
    claude_raw["prompt_tokens"]     * CLAUDE_PRICE_INPUT  / 1_000_000
    + claude_raw["completion_tokens"] * CLAUDE_PRICE_OUTPUT / 1_000_000
)
CLAUDE_BLENDED = claude_cost_total / claude_raw["total_tokens"] * 1_000_000

print(f"Claude Sonnet 4.6  |  {claude_raw['n_ok']:,} requests  "
      f"|  blended ${CLAUDE_BLENDED:.2f}/1M tokens")

# Open-source models
def pretty_log(stem):
    s = stem.replace("llm_calls_", "")
    for x in ["-Instruct-2507", "-Instruct", "-BF16", "-it", "NVIDIA-"]:
        s = s.replace(x, "")
    return s

cost_rows = []
for log_file in sorted(LOGS_DIR.glob("*.jsonl")):
    if "claude" in log_file.name:
        continue
    raw = parse_log(log_file)
    if raw["latency_s"] == 0:
        continue
    tput     = raw["total_tokens"] / raw["latency_s"]
    cost_low  = (1_000_000 / (tput * 3600)) * GPU_RATE_LOW
    cost_high = (1_000_000 / (tput * 3600)) * GPU_RATE_HIGH
    cost_rows.append(dict(
        model=pretty_log(log_file.stem),
        n_ok=raw["n_ok"],
        throughput=tput,
        cost_low=cost_low,
        cost_high=cost_high,
        cost_mid=(cost_low + cost_high) / 2,
        reduction_low=CLAUDE_BLENDED / cost_low,
        reduction_high=CLAUDE_BLENDED / cost_high,
    ))

cost_df = pd.DataFrame(cost_rows).sort_values("cost_low").reset_index(drop=True)

disp = cost_df[["model","n_ok","throughput","cost_low","cost_high","reduction_low","reduction_high"]].copy()
disp.columns = ["Model","N (ok)","Throughput (tok/s)","Cost/1M low","Cost/1M high","Reduction low","Reduction high"]
disp["Throughput (tok/s)"] = disp["Throughput (tok/s)"].map("{:.0f}".format)
disp["Cost/1M low"]  = disp["Cost/1M low"].map("${:.3f}".format)
disp["Cost/1M high"] = disp["Cost/1M high"].map("${:.3f}".format)
disp["Reduction low"]  = disp["Reduction low"].map("{:.1f}x".format)
disp["Reduction high"] = disp["Reduction high"].map("{:.1f}x".format)
display(disp)

In [ ]:
# Bar chart: cost per 1M tokens
fig, ax = plt.subplots(figsize=(10, 4))

labels = cost_df["model"].tolist() + ["Claude Sonnet 4.6"]
mids   = cost_df["cost_mid"].tolist() + [CLAUDE_BLENDED]
errs   = ((cost_df["cost_high"] - cost_df["cost_low"]) / 2).tolist() + [0]
colors = list(sns.color_palette("tab10", n_colors=len(cost_df))) + ["#c0392b"]

x = np.arange(len(labels))
bars = ax.bar(x, mids, yerr=errs, capsize=4, color=colors,
              edgecolor="black", linewidth=0.6,
              error_kw=dict(elinewidth=1.2, ecolor="dimgrey"),
              hatch=[""] * len(cost_df) + ["/"])

for bar, mid, err, rl, rh in zip(
        bars, cost_df["cost_mid"], (cost_df["cost_high"]-cost_df["cost_low"])/2,
        cost_df["reduction_low"], cost_df["reduction_high"]):
    ax.text(bar.get_x() + bar.get_width()/2, mid + err + 0.04,
            f"{rh:.1f}–{rl:.1f}×", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#444")

ax.text(bars[-1].get_x() + bars[-1].get_width()/2, CLAUDE_BLENDED + 0.04,
        f"${CLAUDE_BLENDED:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#c0392b")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("Effective cost per 1M tokens ($)")
ax.set_ylim(0, CLAUDE_BLENDED * 1.2)
ax.set_title("Cost per 1M tokens — open-source evaluators vs Claude Sonnet 4.6 (SWB)",
             fontweight="bold")
ax.legend(handles=[
    mpatches.Patch(color="steelblue", label="Open-source (H100 GPU)"),
    mpatches.Patch(facecolor="#c0392b", hatch="/", edgecolor="black", label="Claude Sonnet 4.6 (API)"),
], loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()

## 5. MOJO cost analysis

For each MAE threshold τ, MOJO routes each rubric to the cheapest open-source model whose bootstrap UCB-MAE ≤ τ, falling back to Claude otherwise.  
Blended cost = mean over rubrics of (escalated ? `CLAUDE_BLENDED` : `cost_mid[assigned_model]`).

In [ ]:
sys.path.insert(0, str(Path("../../figures").resolve()))
from pareto_figures import DATASETS, load_dataset, bootstrap_calibration

# Log pretty names (from pretty_log) → evaluator names (from _sw_pretty / result file stems)
LOG_TO_EVALUATOR = {
    "gemma-4-E2B":        "gemma-4-E2B",
    "gemma-4-E4B":        "gemma-4-E4B",
    "Llama-3.2-3B":       "Llama-3.2-3B",
    "Nemotron-3-Nano-4B": "Nemotron-3-Nano-4B",
    "Qwen3-4B":           "Qwen3-4B",
    "Qwen3.5-4B":         "Qwen3.5-4B",
}

# Build model evaluator-name → cost_mid lookup
MODEL_COST = {
    LOG_TO_EVALUATOR[log]: cost
    for log, cost in zip(cost_df["model"], cost_df["cost_mid"])
    if log in LOG_TO_EVALUATOR
}

swb_cfg = next(c for c in DATASETS if c.name == "story_writing_benchmark")
pairs, errors, evaluator_names = load_dataset(swb_cfg)

cal = bootstrap_calibration(pairs, k=5, alpha=0.05, seed=0)
score_table = (
    cal.pivot_table(index="category", columns="evaluator", values="ucb_mae")
       .reindex(index=swb_cfg.categories, columns=evaluator_names)
)
cat_mae = (
    errors.groupby(["evaluator", "category"])["abs_err"].mean()
          .unstack("evaluator")
          .reindex(index=swb_cfg.categories, columns=evaluator_names)
)

print(f"Rubrics: {len(swb_cfg.categories)} | Evaluators: {evaluator_names}")
print(f"\nCost lookup ($/1M): { {k: round(v,3) for k,v in MODEL_COST.items()} }")

In [ ]:
# SWB score range 0–5; thresholds as fractions of range
THRESHOLDS = [("No fallback", float("inf")),
              ("τ=0.65 (13%)", 0.65),
              ("τ=0.55 (11%)", 0.55),
              ("τ=0.45 (9%)",  0.45),
              ("τ=0.33 (6.5%)", 0.33)]

mojo_rows = []
chosen    = score_table.idxmin(axis=1)
chosen_sc = score_table.min(axis=1)

for label, tau in THRESHOLDS:
    rubric_costs, rubric_maes, n_escalated = [], [], 0
    for rubric in swb_cfg.categories:
        if pd.isna(chosen[rubric]) or chosen_sc[rubric] > tau:
            rubric_costs.append(CLAUDE_BLENDED)
            rubric_maes.append(0.0)
            n_escalated += 1
        else:
            model = chosen[rubric]
            rubric_costs.append(MODEL_COST.get(model, CLAUDE_BLENDED))
            rubric_maes.append(float(cat_mae.loc[rubric, model]))

    blended_cost = float(np.mean(rubric_costs))
    mojo_rows.append(dict(
        label=label,
        n_escalated=n_escalated,
        blended_cost=blended_cost,
        realized_mae=float(np.mean(rubric_maes)),
        reduction=CLAUDE_BLENDED / blended_cost,
    ))

mojo_df = pd.DataFrame(mojo_rows)
mojo_df["Cost/1M"]      = mojo_df["blended_cost"].map("${:.3f}".format)
mojo_df["Reduction"]    = mojo_df["reduction"].map("{:.1f}×".format)
mojo_df["Realized MAE"] = mojo_df["realized_mae"].map("{:.3f}".format)
display(mojo_df[["label","n_escalated","Cost/1M","Reduction","Realized MAE"]]
          .rename(columns={"label":"Threshold","n_escalated":"Rubrics → Claude"}))

no_fb    = mojo_df[mojo_df["n_escalated"] == 0].iloc[0]
tightest = mojo_df[mojo_df["n_escalated"] < len(swb_cfg.categories)].iloc[-1]
print(f"\nFor cost table (Story Writing Benchmark):")
print(f"  MOJO range: ${no_fb['blended_cost']:.2f}–${tightest['blended_cost']:.2f}/1M  "
      f"({no_fb['reduction']:.1f}–{tightest['reduction']:.1f}×)")